# 02. Traffic Volumes EDA
**Task 1.2.2 - Traffic & Active Transport Trends**

### Objective:
- Conduct an EDA on SCATS interval data and historical AADT to understand baseline traffic volumes and turning movements.
- Aggregate traffic and turning movements by hour and day to establish baseline congestion profiles for before and after comparisons.


In [ ]:
import sys
import os
from pathlib import Path

# Path to the .venv in the project root
venv_path = Path("../../.venv")

if not venv_path.exists():
    print("Virtual environment not found. Creating and installing dependencies...")
    !python3 -m venv ../../.venv
    !../../.venv/bin/pip install -r ../../requirements.txt

# Dynamically load venv dependencies into the notebook session
site_packages_dirs = list(venv_path.glob("lib/python*/site-packages"))
if site_packages_dirs:
    site_packages = str(site_packages_dirs[0].resolve())
    if site_packages not in sys.path:
        sys.path.insert(0, site_packages)
        print(f"Dynamically loaded venv dependencies from {site_packages}")


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import yaml
from pathlib import Path

# Setup plotting styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid")

# Load project configuration
config_path = Path("../../config.yaml")
if config_path.exists():
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    print("Project configuration loaded successfully.")
    print("Data Paths:", config.get('paths', {}))
else:
    print("Warning: config.yaml not found at", config_path.resolve())


### 1. Load SCATS Traffic Volume and AADT Data

In [ ]:
# Path to raw SCATS 15-minute interval data
scats_path = Path("../../data/raw/scats_volume_data.csv")

if not scats_path.exists():
    print(f"SCATS data not found at {scats_path}. Generating simulated baseline data for EDA demonstration.")
    # Simulate SCATS data for EDA purposes matching expected schema
    dates = pd.date_range(start="2013-01-01", end="2014-12-31", freq="15min")
    df_scats = pd.DataFrame({
        "timestamp": dates,
        "intersection_id": np.random.choice([1001, 1002, 1003, 1004], size=len(dates)),
        "traffic_volume": np.random.negative_binomial(n=20, p=0.05, size=len(dates)), # Poisson/NB distribution
        "degree_of_saturation": np.random.uniform(0.1, 0.95, size=len(dates))
    })
else:
    df_scats = pd.read_csv(scats_path, parse_dates=["timestamp"])

print(f"Loaded {len(df_scats)} SCATS traffic intervals.")
display(df_scats.head(2))


### 2. Aggregate Traffic and Intersection Turning Movements by Hour/Day

In [ ]:
# Extract temporal features
df_scats["hour"] = df_scats["timestamp"].dt.hour
df_scats["day_of_week"] = df_scats["timestamp"].dt.day_name()
df_scats["year"] = df_scats["timestamp"].dt.year

# Aggregate by hour and day of week to establish profiles
hourly_profile = df_scats.groupby(["intersection_id", "hour"])["traffic_volume"].mean().reset_index()
day_profile = df_scats.groupby(["intersection_id", "day_of_week"])["traffic_volume"].mean().reset_index()

# Sort days of week correctly
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_profile["day_of_week"] = pd.Categorical(day_profile["day_of_week"], categories=day_order, ordered=True)
day_profile = day_profile.sort_values("day_of_week")

print("Aggregated hourly and daily profiles computed.")
display(hourly_profile.head(2))


### 3. Visualize Congestion Profiles and Turning Movements

In [ ]:
fig,axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Hourly Profile
sns.lineplot(
    data=hourly_profile, x="hour", y="traffic_volume", hue="intersection_id", 
    marker="o", palette="Set1", ax=axes[0]
)
axes[0].set_title("Average Hourly Traffic Volumes (Congestion Profile)", fontsize=14)
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Average Traffic Volume (Vehicles / 15-min)")
axes[0].grid(True, linestyle="--", alpha=0.5)

# Plot Day of Week Profile
sns.barplot(
    data=day_profile, x="day_of_week", y="traffic_volume", hue="intersection_id", 
    palette="Set2", ax=axes[1]
)
axes[1].set_title("Average Traffic Volumes by Day of Week", fontsize=14)
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Average Traffic Volume")
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, linestyle="--", alpha=0.5, axis="y")

plt.suptitle("SCATS Baseline Congestion and Traffic Pattern Analysis", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()
